# PhysioNet 2009 MI CSP/FBCSP Baselines

This notebook mirrors the current PhysioNetMI S-JEPA fine-tuning workflow, but replaces the neural network with classical **CSP** and **FBCSP** baselines.

The purpose is to answer a narrow sanity question:

> Are the exact PhysioNet MI windows used by the S-JEPA experiment learnable at all with classical MI decoding?

It keeps the same important choices from the S-JEPA notebook:

- Dataset: `PhysionetMI` through MOABB / Braindecode
- Labels: `left_hand` vs `right_hand`
- Imagined MI only, executed movement excluded
- Subject 88 excluded
- Average reference before preprocessing
- EEG only, resample to 128 Hz, bandpass 0.5--40 Hz
- Same annotation filtering and duration override to 3.2 s
- Same post-window µV scaling
- Same within-subject 5-fold stratified CV with seed 12
- Same artifact style: `config.json`, `cv_results.json`, `subject_metrics.json`, `global_metrics.json`, `run_metadata.json`, and `run.log`

# 1. Setup

## 1.1. Import Libraries

In [ ]:
import os
import random
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime
import math
import warnings

import numpy as np
from numpy import multiply

from braindecode.datasets import MOABBDataset, BaseConcatDataset
from braindecode.datautil import load_concat_dataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    create_windows_from_events,
)

import mne
from mne.decoding import CSP
mne.set_log_level("WARNING")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.base import clone

import builtins

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("All imports loaded successfully.")

## 1.2. Runtime & Path Validation

In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().parent
print(f"\nWorking directory: {WORKING_DIR}")

# 2. Configuration

## 2.1. Config

In [ ]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "physionet-2009-csp-fbcsp-baselines"),

    # Dataset source
    # - "auto": use preprocessed_concat_dir if it exists; otherwise build from MOABB and save
    # - "moabb": force build from MOABB and save
    # - "concat": force load an already saved Braindecode concat dataset
    "dataset_source": "auto",
    "dataset_name": "PhysionetMI",

    # This is the cached raw-concat path from the current S-JEPA run metadata.
    # If this path does not exist on the current machine, the notebook falls back to the deterministic default cache path.
    "preprocessed_concat_dir": "/home/vegorov/Repos/eeg_jepa_research/preprocessed_concat_datasets/PhysionetMI_MI_048eba1d_subjects_224f4495",

    # PhysioNetMI selection
    "labels_to_keep": ["left_hand", "right_hand"],
    "exclude_subjects": [88],
    "subjects_to_use": None,  # None = auto-select all valid PhysioNet subjects
    "moabb_dataset_kwargs": {
        "imagined": True,
        "executed": False,
    },

    # Reproducibility
    "seed": 12,
    "set_seed": True,

    # Base preprocessing: keep this aligned with the S-JEPA run
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,

    # Paradigm/windowing settings: keep aligned with current PhysioNet S-JEPA run
    "paradigm": "MI",
    "base_trial_duration_s": 3.0,
    "target_window_duration_s": 3.2,

    # Cross-validation
    "cv_folds": 5,

    # Baselines to run
    "run_models": ["CSP_LDA", "FBCSP_LDA"],

    # CSP baseline settings.
    # The raw data has already been preprocessed at 0.5-40 Hz. This optional feature-stage filter
    # focuses CSP on the usual MI mu/beta range while keeping the same windows/splits.
    "csp": {
        "apply_feature_filter": True,
        "l_freq": 8.0,
        "h_freq": 30.0,
        "n_components": 4,
        "reg": "ledoit_wolf",
        "log": True,
        "norm_trace": False,
    },

    # FBCSP settings: 4-Hz subbands from 4 to 40 Hz.
    "fbcsp": {
        "bands": [
            [4.0, 8.0],
            [8.0, 12.0],
            [12.0, 16.0],
            [16.0, 20.0],
            [20.0, 24.0],
            [24.0, 28.0],
            [28.0, 32.0],
            [32.0, 36.0],
            [36.0, 40.0],
        ],
        "n_components": 4,
        "reg": "ledoit_wolf",
        "log": True,
        "norm_trace": False,
    },

    # Classical classifier used after CSP/FBCSP features.
    # Options: "lda" or "linear_svm".
    "classifier": "lda",
    "linear_svm_C": 1.0,

    # Feature-stage filtering. IIR is used here to avoid very long FIR filters on short per-fold windows.
    # This does not change the base preprocessing cache; it only applies inside CSP/FBCSP feature extraction.
    "feature_filter_method": "iir",
    "feature_iir_params": {"order": 5, "ftype": "butter"},
}

LOG_COMPACT = True
PRINT_FOLD_CLASS_COUNTS = False

In [ ]:
PARADIGM_CONFIGS = {
    "MI": {
        "n_classes": 2,
        "base_trial_duration_s": CONFIG["base_trial_duration_s"],
        "target_window_duration_s": CONFIG["target_window_duration_s"],
    },
}

PHYSIONET_ALL_SUBJECTS = list(range(1, 110))
PHYSIONET_VALID_SUBJECTS = [s for s in PHYSIONET_ALL_SUBJECTS if s not in set(CONFIG["exclude_subjects"])]
DEFAULT_SUBJECTS = PHYSIONET_VALID_SUBJECTS

CLASSES_MAPPING = {label: idx for idx, label in enumerate(CONFIG["labels_to_keep"])}

## 2.2. Create Artifact Directory

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["paradigm"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifact directory: {ARTIFACT_DIR}")

## 2.3. Initialize Logger

In [ ]:
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(arg) for arg in args)

    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text)
            if flush:
                sys.__stdout__.flush()
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print
print(f"Logging to: {LOG_PATH}")

## 2.4. Deterministic Seeding

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None

if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None. Please specify a seed.")
    seed_everything(BASE_SEED)
    print(f"Seed control enabled: {CONFIG['set_seed']}")
    print(f"Base seed: {BASE_SEED}")
else:
    print("Seed control disabled (CONFIG['set_seed'] = False)")

## 2.5. Save Configuration

In [ ]:
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(json.dumps(CONFIG, indent=2, default=str))

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Saved config to: {config_path}")

# 3. Load and Prepare Data

## 3.1. Derived Constants

In [ ]:
def current_paradigm_config(paradigm, paradigm_configs):
    if paradigm != "MI":
        raise ValueError("This notebook currently supports PhysioNet motor imagery only (paradigm='MI').")
    return (
        paradigm_configs[paradigm]["n_classes"],
        paradigm_configs[paradigm]["base_trial_duration_s"],
        paradigm_configs[paradigm]["target_window_duration_s"],
    )

def compute_epoch_window(sfreq, target_trial_duration_s):
    sfreq = float(sfreq)
    target_trial_duration_s = float(target_trial_duration_s)
    window_size_samples = int(math.floor(target_trial_duration_s * sfreq))

    print("Epoch window configuration:")
    print(f"  Target sfreq:                {sfreq:.0f} Hz")
    print(f"  Requested duration:          {target_trial_duration_s:.2f} s")
    print(f"  Expected window samples:     {window_size_samples}")

    return window_size_samples

TARGET_N_CLASSES, BASE_TRIAL_DURATION_S, TARGET_TRIAL_DURATION_S = current_paradigm_config(
    CONFIG["paradigm"],
    PARADIGM_CONFIGS,
)

WINDOW_SAMPLES = compute_epoch_window(
    sfreq=CONFIG["sfreq"],
    target_trial_duration_s=TARGET_TRIAL_DURATION_S,
)

In [ ]:
SUBJECTS = CONFIG["subjects_to_use"] if CONFIG["subjects_to_use"] is not None else DEFAULT_SUBJECTS
SUBJECTS = [int(s) for s in SUBJECTS]

invalid_subjects = sorted(set(SUBJECTS).intersection(set(CONFIG["exclude_subjects"])))
if invalid_subjects:
    raise ValueError(
        f"The following subject IDs are excluded by config and cannot be loaded together: {invalid_subjects}"
    )

label_fragment = "_".join(CONFIG["labels_to_keep"])
label_hash = hashlib.md5(label_fragment.encode()).hexdigest()[:8]

subject_fragment = "_".join(str(s) for s in SUBJECTS)
subject_hash = hashlib.md5(subject_fragment.encode()).hexdigest()[:8]

DEFAULT_PREPROCESSED_CONCAT_DIR = WORKING_DIR / "preprocessed_concat_datasets" / (
    f"{CONFIG['dataset_name']}_{CONFIG['paradigm']}_{label_hash}_subjects_{subject_hash}"
)

provided_concat_dir = Path(CONFIG["preprocessed_concat_dir"]) if CONFIG["preprocessed_concat_dir"] else None

if provided_concat_dir is not None and provided_concat_dir.exists() and provided_concat_dir.is_dir():
    PREPROCESSED_CONCAT_DIR = provided_concat_dir
else:
    PREPROCESSED_CONCAT_DIR = DEFAULT_PREPROCESSED_CONCAT_DIR

PREPROCESSED_CONCAT_DATASETS_EXISTS = PREPROCESSED_CONCAT_DIR.exists() and PREPROCESSED_CONCAT_DIR.is_dir()

print(f"\nProvided concat dir:              {provided_concat_dir}")
print(f"Default concat dir:               {DEFAULT_PREPROCESSED_CONCAT_DIR}")
print(f"Selected concat dir:              {PREPROCESSED_CONCAT_DIR}")
print(f"Preprocessed concat exists:       {PREPROCESSED_CONCAT_DATASETS_EXISTS}")

if CONFIG["dataset_source"] not in {"auto", "moabb", "concat"}:
    raise ValueError("CONFIG['dataset_source'] must be 'auto', 'moabb', or 'concat'.")

if CONFIG["dataset_source"] == "auto":
    EFFECTIVE_DATASET_SOURCE = "concat" if PREPROCESSED_CONCAT_DATASETS_EXISTS else "moabb"
else:
    EFFECTIVE_DATASET_SOURCE = CONFIG["dataset_source"]

print(f"Effective dataset source:         {EFFECTIVE_DATASET_SOURCE}")

## 3.2. Load Data

In [ ]:
if EFFECTIVE_DATASET_SOURCE == "concat" and not PREPROCESSED_CONCAT_DATASETS_EXISTS:
    raise FileNotFoundError(
        "CONFIG requested concat loading, but the selected preprocessed concat directory does not exist."
    )

if EFFECTIVE_DATASET_SOURCE == "moabb":
    DATASET_NAME = CONFIG["dataset_name"]
    DATASET_KWARGS = dict(CONFIG["moabb_dataset_kwargs"])

    print(f"\nBuilding MOABBDataset for {DATASET_NAME}...")
    print(f"  Paradigm:              {CONFIG['paradigm']}")
    print(f"  Subject IDs:           {SUBJECTS}")
    print(f"  Dataset kwargs:        {DATASET_KWARGS}")
    print(f"  Labels to keep:        {CONFIG['labels_to_keep']}")
    print(f"  Excluded subject IDs:  {CONFIG['exclude_subjects']}")

    DATASET = MOABBDataset(
        dataset_name=DATASET_NAME,
        subject_ids=SUBJECTS,
        dataset_kwargs=DATASET_KWARGS,
    )

    unique_labels = sorted(
        {
            str(desc)
            for ds in DATASET.datasets
            for desc in np.asarray(ds.raw.annotations.description).astype(str)
        }
    )

    print(f"  Recordings loaded:     {len(DATASET.datasets)}")
    print(f"  Raw labels detected:   {unique_labels}")
else:
    print("\nUsing existing preprocessed concat dataset only.")
    print(f"  Concat dir:            {PREPROCESSED_CONCAT_DIR}")
    print(f"  Subject IDs:           {SUBJECTS}")
    print(f"  Labels to keep:        {CONFIG['labels_to_keep']}")

## 3.3. Preprocessing

In [ ]:
if EFFECTIVE_DATASET_SOURCE == "moabb":
    for ds in DATASET.datasets:
        mne.set_eeg_reference(ds.raw, ref_channels="average", copy=False, verbose=True)

    print("Applying Braindecode preprocessors to MOABBDataset...")
    preprocess(
        DATASET,
        [
            Preprocessor("pick_types", eeg=True, meg=False, stim=False),
            Preprocessor("resample", sfreq=CONFIG["sfreq"]),
            Preprocessor(
                "filter",
                l_freq=CONFIG["bandpass_low"],
                h_freq=CONFIG["bandpass_high"],
            ),
        ],
        n_jobs=1,
    )

    DATASET.save(PREPROCESSED_CONCAT_DIR, overwrite=True)
    print(f"Saved preprocessed raw concat dataset to: {PREPROCESSED_CONCAT_DIR}")

In [ ]:
LOADED_DATASET = load_concat_dataset(
    PREPROCESSED_CONCAT_DIR,
    preload=True,
    target_name=None,
    ids_to_load=None,
)
print(f"Loaded raw concat dataset from: {PREPROCESSED_CONCAT_DIR}")

first_raw = LOADED_DATASET.datasets[0].raw
subject_ids = sorted(set(str(ds.description["subject"]) for ds in LOADED_DATASET.datasets))
all_annotation_labels = sorted(
    {
        str(desc)
        for ds in LOADED_DATASET.datasets
        for desc in np.asarray(ds.raw.annotations.description).astype(str)
    }
)

print("\nPost-load raw validation")
print(f"  Recordings in LOADED_DATASET:   {len(LOADED_DATASET.datasets)}")
print(f"  Unique subject IDs:             {subject_ids}")
print(f"  sfreq (first rec):              {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:              {len(first_raw.ch_names)}")
print(f"  Annotation labels present:      {all_annotation_labels}")
print(f"  Labels selected for training:   {CONFIG['labels_to_keep']}")

## 3.4. Windowing

In [ ]:
def summarize_annotation_counts(dataset):
    counts = {}
    for ds in dataset.datasets:
        descriptions = np.asarray(ds.raw.annotations.description).astype(str)
        for desc in descriptions:
            counts[desc] = counts.get(desc, 0) + 1
    return dict(sorted(counts.items()))


def filter_annotations_by_description(dataset, descriptions_to_keep):
    keep_set = set(str(d) for d in descriptions_to_keep)
    total_before = 0
    total_after = 0

    for ds in dataset.datasets:
        annotations = ds.raw.annotations
        if len(annotations) == 0:
            continue

        descriptions = np.asarray(annotations.description).astype(str)
        mask = np.isin(descriptions, list(keep_set))
        total_before += len(annotations)
        total_after += int(np.sum(mask))

        filtered_annotations = mne.Annotations(
            onset=np.asarray(annotations.onset)[mask],
            duration=np.asarray(annotations.duration)[mask],
            description=descriptions[mask].tolist(),
            orig_time=annotations.orig_time,
        )
        ds.raw.set_annotations(filtered_annotations)

    return total_before, total_after


def update_annotation_durations(dataset, target_duration_s):
    updated_annotations = 0
    descriptions_to_keep = CONFIG["labels_to_keep"]

    for ds in dataset.datasets:
        annotations = ds.raw.annotations
        if len(annotations) == 0:
            continue

        descriptions = np.asarray(annotations.description).astype(str)
        mask = np.isin(descriptions, descriptions_to_keep)

        if not np.any(mask):
            continue

        durations = np.asarray(annotations.duration, dtype=float).copy()
        durations[mask] = float(target_duration_s)
        annotations.duration = durations
        updated_annotations += int(np.sum(mask))

    return updated_annotations


def build_windows_dataset(dataset, target_duration_s, mapping):
    before_counts = summarize_annotation_counts(dataset)
    print(f"Annotation counts before filtering: {before_counts}")

    total_before, total_after = filter_annotations_by_description(
        dataset,
        descriptions_to_keep=list(mapping.keys()),
    )
    print(f"Filtered annotations: kept {total_after} / {total_before}")

    after_counts = summarize_annotation_counts(dataset)
    print(f"Annotation counts after filtering:  {after_counts}")

    updated_count = update_annotation_durations(
        dataset,
        target_duration_s=target_duration_s,
    )
    print(f"Updated annotation durations:       {updated_count}")

    datasets_with_events = [ds for ds in dataset.datasets if len(ds.raw.annotations) > 0]
    dropped_recordings = len(dataset.datasets) - len(datasets_with_events)
    if dropped_recordings > 0:
        print(f"Dropped recordings with zero kept events before windowing: {dropped_recordings}")

    if len(datasets_with_events) == 0:
        raise RuntimeError(
            "No recordings contain the selected labels after filtering. "
            "Check CONFIG['labels_to_keep'] and the loaded dataset labels."
        )

    filtered_dataset = BaseConcatDataset(datasets_with_events)

    windows_dataset = create_windows_from_events(
        filtered_dataset,
        trial_start_offset_samples=0,
        mapping=mapping,
        window_size_samples=None,
        window_stride_samples=None,
        drop_last_window=False,
        preload=True,
    )

    return windows_dataset

In [ ]:
WINDOWS_DATASET = build_windows_dataset(
    LOADED_DATASET,
    target_duration_s=TARGET_TRIAL_DURATION_S,
    mapping=CLASSES_MAPPING,
)

def scale_volts_to_microvolts(data):
    return multiply(data, 1e6)

preprocess(
    WINDOWS_DATASET,
    [Preprocessor(scale_volts_to_microvolts)],
    n_jobs=1,
)

sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
y0 = sample0[1]
print(f"Window shape: {tuple(x0.shape)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")
print(f"Class mapping: {CLASSES_MAPPING}")

ALL_LABELS = np.asarray([int(WINDOWS_DATASET[i][1]) for i in range(len(WINDOWS_DATASET))], dtype=np.int64)
all_counts = np.bincount(ALL_LABELS, minlength=TARGET_N_CLASSES)
print(f"Global class counts: {all_counts.tolist()}")
print(f"Chance level ({TARGET_N_CLASSES}-class): {1.0 / TARGET_N_CLASSES:.2f}")

## 3.5. Get Subject Windows

In [ ]:
def get_subject_windows(windows_dataset, subjects):
    target_subjects = {str(s) for s in subjects}
    subject_to_indices = {str(s): [] for s in subjects}

    for idx, ds in enumerate(windows_dataset.datasets):
        subject_id = str(ds.description.get("subject"))
        if subject_id in target_subjects:
            subject_to_indices[subject_id].append(idx)

    missing_subjects = [sid for sid, idxs in subject_to_indices.items() if len(idxs) == 0]
    if missing_subjects:
        print(f"Subjects with no windows after filtering: {missing_subjects}")

    subject_windows = {}
    for sid in sorted(subject_to_indices.keys(), key=int):
        idxs = subject_to_indices[sid]
        if len(idxs) == 0:
            continue
        subject_windows[sid] = BaseConcatDataset([windows_dataset.datasets[i] for i in idxs])

    if len(subject_windows) == 0:
        raise RuntimeError("No subject windows were created. Ensure windows carry subject metadata and labels exist.")

    return subject_windows

SUBJECT_WINDOWS = get_subject_windows(WINDOWS_DATASET, SUBJECTS)

In [ ]:
def summarize_subject_windows(subject_windows, n_classes):
    print("Summarizing per-subject windows...")
    for subject_id in sorted(subject_windows, key=int):
        ds = subject_windows[subject_id]
        y = np.asarray([int(ds[i][1]) for i in range(len(ds))], dtype=np.int64)
        counts = np.bincount(y, minlength=n_classes)
        x = ds[0][0]
        print(
            f"  Subject {subject_id}: n_windows={len(ds)}, "
            f"window_shape={tuple(x.shape)}, class_counts={counts.tolist()}"
        )

summarize_subject_windows(SUBJECT_WINDOWS, TARGET_N_CLASSES)

# 4. CSP/FBCSP Utilities

## 4.1. Extract Arrays

In [ ]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)


def dataset_to_X_y(dataset):
    X = []
    y = []
    for i in range(len(dataset)):
        sample = dataset[i]
        X.append(np.asarray(sample[0], dtype=np.float64))
        y.append(int(sample[1]))
    return np.stack(X, axis=0), np.asarray(y, dtype=np.int64)


def make_fold_splits(y, n_folds, n_classes):
    y = np.asarray(y, dtype=np.int64)
    counts = np.bincount(y, minlength=n_classes)
    min_class_count = int(np.min(counts))

    if min_class_count < n_folds:
        raise ValueError(
            f"Cannot run {n_folds}-fold stratified CV because the smallest class has "
            f"only {min_class_count} samples. Class counts: {counts.tolist()}"
        )

    splitter = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=BASE_SEED,
    )

    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y)), y), start=1):
        folds.append({"fold_id": fold_id, "idx_train": train_idx, "idx_test": test_idx})
    return folds

## 4.2. Filtering and Classifiers

In [ ]:
def bandpass_epochs(X, l_freq, h_freq, sfreq):
    """Filter epoched data with shape (n_trials, n_channels, n_times)."""
    return mne.filter.filter_data(
        X.copy(),
        sfreq=float(sfreq),
        l_freq=float(l_freq) if l_freq is not None else None,
        h_freq=float(h_freq) if h_freq is not None else None,
        method=CONFIG["feature_filter_method"],
        iir_params=dict(CONFIG["feature_iir_params"]),
        n_jobs=1,
        verbose=False,
    )


def make_classifier():
    classifier_name = CONFIG["classifier"].lower()
    if classifier_name == "lda":
        # shrinkage LDA is more stable for tiny within-subject PhysioNet folds
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if classifier_name == "linear_svm":
        return SVC(
            kernel="linear",
            C=float(CONFIG["linear_svm_C"]),
            class_weight="balanced",
            probability=True,
            random_state=BASE_SEED,
        )
    raise ValueError(f"Unknown classifier: {CONFIG['classifier']}")


def get_score_values(estimator, X_features_or_raw):
    """Return a continuous class-1 score for ROC-AUC when available."""
    try:
        if hasattr(estimator, "decision_function"):
            scores = estimator.decision_function(X_features_or_raw)
            scores = np.asarray(scores)
            if scores.ndim == 2 and scores.shape[1] > 1:
                return scores[:, 1]
            return scores.reshape(-1)
        if hasattr(estimator, "predict_proba"):
            probs = estimator.predict_proba(X_features_or_raw)
            return np.asarray(probs)[:, 1]
    except Exception:
        return None
    return None


def safe_roc_auc(y_true, score_values):
    if score_values is None:
        return None
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return None
    try:
        return float(roc_auc_score(y_true, score_values))
    except Exception:
        return None

## 4.3. CSP and FBCSP Fit/Predict

In [ ]:
def make_csp_transformer(n_components):
    return CSP(
        n_components=int(n_components),
        reg=CONFIG["csp"]["reg"],
        log=bool(CONFIG["csp"]["log"]),
        norm_trace=bool(CONFIG["csp"]["norm_trace"]),
    )


def fit_predict_csp(X_train, y_train, X_test):
    csp_cfg = CONFIG["csp"]
    if csp_cfg["apply_feature_filter"]:
        X_train_use = bandpass_epochs(
            X_train,
            l_freq=csp_cfg["l_freq"],
            h_freq=csp_cfg["h_freq"],
            sfreq=CONFIG["sfreq"],
        )
        X_test_use = bandpass_epochs(
            X_test,
            l_freq=csp_cfg["l_freq"],
            h_freq=csp_cfg["h_freq"],
            sfreq=CONFIG["sfreq"],
        )
        band = [float(csp_cfg["l_freq"]), float(csp_cfg["h_freq"])]
    else:
        X_train_use = X_train
        X_test_use = X_test
        band = [float(CONFIG["bandpass_low"]), float(CONFIG["bandpass_high"])]

    n_components = min(int(csp_cfg["n_components"]), X_train_use.shape[1])

    estimator = Pipeline(
        steps=[
            ("csp", make_csp_transformer(n_components)),
            ("scaler", StandardScaler()),
            ("classifier", make_classifier()),
        ]
    )

    estimator.fit(X_train_use, y_train)
    y_pred = estimator.predict(X_test_use)
    score_values = get_score_values(estimator, X_test_use)

    extra = {
        "feature_band": band,
        "n_csp_components": n_components,
        "n_features": n_components,
    }
    return y_pred, score_values, extra


def fit_predict_fbcsp(X_train, y_train, X_test):
    fb_cfg = CONFIG["fbcsp"]
    train_features = []
    test_features = []
    used_bands = []

    for band in fb_cfg["bands"]:
        l_freq, h_freq = float(band[0]), float(band[1])
        X_train_band = bandpass_epochs(X_train, l_freq=l_freq, h_freq=h_freq, sfreq=CONFIG["sfreq"])
        X_test_band = bandpass_epochs(X_test, l_freq=l_freq, h_freq=h_freq, sfreq=CONFIG["sfreq"])

        n_components = min(int(fb_cfg["n_components"]), X_train_band.shape[1])
        csp = CSP(
            n_components=n_components,
            reg=fb_cfg["reg"],
            log=bool(fb_cfg["log"]),
            norm_trace=bool(fb_cfg["norm_trace"]),
        )

        train_features.append(csp.fit_transform(X_train_band, y_train))
        test_features.append(csp.transform(X_test_band))
        used_bands.append([l_freq, h_freq])

    if len(train_features) == 0:
        raise RuntimeError("No FBCSP bands produced features.")

    X_train_features = np.concatenate(train_features, axis=1)
    X_test_features = np.concatenate(test_features, axis=1)

    estimator = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", make_classifier()),
        ]
    )
    estimator.fit(X_train_features, y_train)
    y_pred = estimator.predict(X_test_features)
    score_values = get_score_values(estimator, X_test_features)

    extra = {
        "feature_bands": used_bands,
        "n_bands": len(used_bands),
        "n_csp_components_per_band": int(fb_cfg["n_components"]),
        "n_features": int(X_train_features.shape[1]),
    }
    return y_pred, score_values, extra

# 5. Cross-Validation

## 5.1. Single Fold Runner

In [ ]:
def run_single_fold(model_name, subject_id, X, y, idx_train, idx_test, fold_id):
    X_train = X[idx_train]
    X_test = X[idx_test]
    y_train = y[idx_train]
    y_test = y[idx_test]

    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    if PRINT_FOLD_CLASS_COUNTS:
        print(
            f"    Fold {fold_id}: train={len(idx_train)} {train_counts.tolist()} | "
            f"test={len(idx_test)} {test_counts.tolist()}"
        )

    if model_name == "CSP_LDA":
        y_pred, score_values, extra = fit_predict_csp(X_train, y_train, X_test)
    elif model_name == "FBCSP_LDA":
        y_pred, score_values, extra = fit_predict_fbcsp(X_train, y_train, X_test)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    acc = float(accuracy_score(y_test, y_pred))
    bal_acc = float(balanced_accuracy_score(y_test, y_pred))
    roc_auc = safe_roc_auc(y_test, score_values)
    cm = confusion_matrix(y_test, y_pred, labels=list(range(TARGET_N_CLASSES))).tolist()
    pred_hist = np.bincount(np.asarray(y_pred, dtype=np.int64), minlength=TARGET_N_CLASSES).tolist()

    return {
        "subject_id": str(subject_id),
        "fold_id": int(fold_id),
        "paradigm": CONFIG["paradigm"],
        "model_name": model_name,
        "classifier": CONFIG["classifier"],
        "n_train": int(len(idx_train)),
        "n_test": int(len(idx_test)),
        "train_class_counts": train_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "roc_auc": roc_auc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
        **extra,
    }

## 5.2. Subject CV Runner

In [ ]:
def run_subject_cv(model_name, subject_id, subject_dataset, n_classes, cv_folds):
    X, y = dataset_to_X_y(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\n[{model_name}] Subject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")

    folds = make_fold_splits(y, n_folds=cv_folds, n_classes=n_classes)
    results = []

    for fold in folds:
        result = run_single_fold(
            model_name=model_name,
            subject_id=subject_id,
            X=X,
            y=y,
            idx_train=fold["idx_train"],
            idx_test=fold["idx_test"],
            fold_id=fold["fold_id"],
        )
        results.append(result)

    acc_values = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bal_acc_values = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    roc_auc_values = [r["roc_auc"] for r in results if r["roc_auc"] is not None]

    mean_acc = float(np.mean(acc_values)) if len(acc_values) > 0 else None
    std_acc = float(np.std(acc_values)) if len(acc_values) > 0 else None
    mean_bal_acc = float(np.mean(bal_acc_values)) if len(bal_acc_values) > 0 else None
    std_bal_acc = float(np.std(bal_acc_values)) if len(bal_acc_values) > 0 else None
    mean_roc_auc = float(np.mean(roc_auc_values)) if len(roc_auc_values) > 0 else None
    std_roc_auc = float(np.std(roc_auc_values)) if len(roc_auc_values) > 0 else None

    acc_str = f"{mean_acc:.4f}±{std_acc:.4f}" if mean_acc is not None and std_acc is not None else "N/A"
    bal_acc_str = f"{mean_bal_acc:.4f}±{std_bal_acc:.4f}" if mean_bal_acc is not None and std_bal_acc is not None else "N/A"
    roc_auc_str = f"{mean_roc_auc:.4f}±{std_roc_auc:.4f}" if mean_roc_auc is not None and std_roc_auc is not None else "N/A"

    print(f"  Subject {subject_id} summary: acc={acc_str}  bal_acc={bal_acc_str}  roc_auc={roc_auc_str}")
    return results

## 5.3. Run All Subjects

In [ ]:
print("=" * 70)
print("STARTING CSP/FBCSP CROSS-VALIDATION")
print("=" * 70)
print(f"Models: {CONFIG['run_models']}")
print(f"Subjects: {len(SUBJECT_WINDOWS)}")
print(f"CV folds: {CONFIG['cv_folds']}")

ALL_MODEL_RESULTS = {}

for model_name in CONFIG["run_models"]:
    print("\n" + "=" * 70)
    print(f"RUNNING MODEL: {model_name}")
    print("=" * 70)

    fold_results = []
    for sid in sorted(SUBJECT_WINDOWS.keys(), key=int):
        subject_results = run_subject_cv(
            model_name=model_name,
            subject_id=sid,
            subject_dataset=SUBJECT_WINDOWS[sid],
            n_classes=TARGET_N_CLASSES,
            cv_folds=CONFIG["cv_folds"],
        )
        fold_results.extend(subject_results)

    ALL_MODEL_RESULTS[model_name] = fold_results
    print(f"\nCompleted {model_name}: {len(fold_results)} folds")

print("\nAll requested CSP/FBCSP models completed.")

# 6. Results

## 6.1. Aggregate Metrics

In [ ]:
def aggregate_fold_results(fold_results):
    subject_metrics = {}

    for result in fold_results:
        sid = result["subject_id"]
        if sid not in subject_metrics:
            subject_metrics[sid] = {
                "accuracies": [],
                "balanced_accuracies": [],
                "roc_aucs": [],
            }
        subject_metrics[sid]["accuracies"].append(result["accuracy"])
        subject_metrics[sid]["balanced_accuracies"].append(result["balanced_accuracy"])
        subject_metrics[sid]["roc_aucs"].append(result.get("roc_auc"))

    for sid, m in subject_metrics.items():
        acc_values = [v for v in m["accuracies"] if v is not None]
        bal_acc_values = [v for v in m["balanced_accuracies"] if v is not None]
        roc_auc_values = [v for v in m["roc_aucs"] if v is not None]

        m["mean_accuracy"] = float(np.mean(acc_values)) if len(acc_values) > 0 else None
        m["std_accuracy"] = float(np.std(acc_values)) if len(acc_values) > 0 else None
        m["mean_balanced_accuracy"] = float(np.mean(bal_acc_values)) if len(bal_acc_values) > 0 else None
        m["std_balanced_accuracy"] = float(np.std(bal_acc_values)) if len(bal_acc_values) > 0 else None
        m["mean_roc_auc"] = float(np.mean(roc_auc_values)) if len(roc_auc_values) > 0 else None
        m["std_roc_auc"] = float(np.std(roc_auc_values)) if len(roc_auc_values) > 0 else None

    all_accs = [r["accuracy"] for r in fold_results if r["accuracy"] is not None]
    all_bal_accs = [r["balanced_accuracy"] for r in fold_results if r["balanced_accuracy"] is not None]
    all_roc_aucs = [r["roc_auc"] for r in fold_results if r.get("roc_auc") is not None]

    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if len(all_accs) > 0 else None,
        "std_accuracy": float(np.std(all_accs)) if len(all_accs) > 0 else None,
        "mean_balanced_accuracy": float(np.mean(all_bal_accs)) if len(all_bal_accs) > 0 else None,
        "std_balanced_accuracy": float(np.std(all_bal_accs)) if len(all_bal_accs) > 0 else None,
        "mean_roc_auc": float(np.mean(all_roc_aucs)) if len(all_roc_aucs) > 0 else None,
        "std_roc_auc": float(np.std(all_roc_aucs)) if len(all_roc_aucs) > 0 else None,
        "n_subjects": len(subject_metrics),
        "n_folds_total": len(fold_results),
    }

    return subject_metrics, global_metrics

SUBJECT_METRICS = {}
GLOBAL_METRICS = {}

for model_name, fold_results in ALL_MODEL_RESULTS.items():
    subject_metrics, global_metrics = aggregate_fold_results(fold_results)
    SUBJECT_METRICS[model_name] = subject_metrics
    GLOBAL_METRICS[model_name] = global_metrics

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)

for model_name in CONFIG["run_models"]:
    gm = GLOBAL_METRICS[model_name]
    acc_str = f"{gm['mean_accuracy']:.4f}±{gm['std_accuracy']:.4f}" if gm["mean_accuracy"] is not None and gm["std_accuracy"] is not None else "N/A"
    bal_acc_str = f"{gm['mean_balanced_accuracy']:.4f}±{gm['std_balanced_accuracy']:.4f}" if gm["mean_balanced_accuracy"] is not None and gm["std_balanced_accuracy"] is not None else "N/A"
    roc_auc_str = f"{gm['mean_roc_auc']:.4f}±{gm['std_roc_auc']:.4f}" if gm["mean_roc_auc"] is not None and gm["std_roc_auc"] is not None else "N/A"
    print(f"  {model_name}: acc={acc_str}  bal_acc={bal_acc_str}  roc_auc={roc_auc_str}  folds={gm['n_folds_total']}")

print("=" * 70)

## 6.2. Prediction Collapse Diagnostics

In [ ]:
def summarize_prediction_collapse(fold_results):
    total = len(fold_results)
    collapsed = 0
    majority_only = {str(c): 0 for c in range(TARGET_N_CLASSES)}

    for r in fold_results:
        hist = r.get("prediction_histogram", [])
        nonzero = sum(1 for v in hist if v > 0)
        if nonzero <= 1:
            collapsed += 1
            if len(hist) > 0:
                majority_only[str(int(np.argmax(hist)))] += 1

    return {
        "n_folds": total,
        "n_collapsed_single_class": collapsed,
        "collapsed_fraction": float(collapsed / total) if total > 0 else None,
        "single_class_prediction_counts": majority_only,
    }

COLLAPSE_DIAGNOSTICS = {}
for model_name, fold_results in ALL_MODEL_RESULTS.items():
    COLLAPSE_DIAGNOSTICS[model_name] = summarize_prediction_collapse(fold_results)

print("Prediction collapse diagnostics")
print(json.dumps(COLLAPSE_DIAGNOSTICS, indent=2))

## 6.3. Save Outputs

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(ALL_MODEL_RESULTS, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
print(f"Subject metrics saved to: {subject_metrics_path}")

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2)
print(f"Global metrics saved to:  {global_metrics_path}")

collapse_path = ARTIFACT_DIR / "collapse_diagnostics.json"
with open(collapse_path, "w") as f:
    json.dump(COLLAPSE_DIAGNOSTICS, f, indent=2)
print(f"Collapse diagnostics saved to: {collapse_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "dataset_name": CONFIG["dataset_name"],
    "dataset_source": CONFIG["dataset_source"],
    "effective_dataset_source": EFFECTIVE_DATASET_SOURCE,
    "preprocessed_concat_dir": str(PREPROCESSED_CONCAT_DIR),
    "labels_to_keep": list(CONFIG["labels_to_keep"]),
    "excluded_subjects": list(CONFIG["exclude_subjects"]),
    "paradigm": CONFIG["paradigm"],
    "subjects": [str(s) for s in SUBJECTS],
    "run_models": list(CONFIG["run_models"]),
    "cv_seed": BASE_SEED,
    "global_metrics": GLOBAL_METRICS,
    "collapse_diagnostics": COLLAPSE_DIAGNOSTICS,
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)
print(f"Run metadata saved to:    {run_metadata_path}")

print(f"\nAll artifacts in: {ARTIFACT_DIR}")

## 6.4. Final Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:    {RUN_ID}")
print(f"Artifacts: {ARTIFACT_DIR}")
print()
print("Configuration:")
print(f"  Dataset name:           {CONFIG['dataset_name']}")
print(f"  Dataset source:         {CONFIG['dataset_source']} -> {EFFECTIVE_DATASET_SOURCE}")
print(f"  Subjects:               {len(SUBJECTS)} subjects")
print(f"  Excluded subjects:      {list(CONFIG['exclude_subjects'])}")
print(f"  Labels kept:            {list(CONFIG['labels_to_keep'])}")
print(f"  Paradigm:               {CONFIG['paradigm']}")
print(f"  Bandpass:               {CONFIG['bandpass_low']}-{CONFIG['bandpass_high']} Hz")
print(f"  Resample:               {CONFIG['sfreq']} Hz")
print(f"  Window duration:        {TARGET_TRIAL_DURATION_S} s")
print(f"  CV folds:               {CONFIG['cv_folds']}")
print(f"  Classifier:             {CONFIG['classifier']}")
print()
print("Results:")
for model_name in CONFIG["run_models"]:
    gm = GLOBAL_METRICS[model_name]
    print(f"  {model_name}:")
    if gm["mean_accuracy"] is not None and gm["std_accuracy"] is not None:
        print(f"    Mean Accuracy:          {gm['mean_accuracy']:.4f} ± {gm['std_accuracy']:.4f}")
    else:
        print("    Mean Accuracy:          N/A")

    if gm["mean_balanced_accuracy"] is not None and gm["std_balanced_accuracy"] is not None:
        print(f"    Mean Balanced Accuracy: {gm['mean_balanced_accuracy']:.4f} ± {gm['std_balanced_accuracy']:.4f}")
    else:
        print("    Mean Balanced Accuracy: N/A")

    if gm["mean_roc_auc"] is not None and gm["std_roc_auc"] is not None:
        print(f"    Mean ROC-AUC:           {gm['mean_roc_auc']:.4f} ± {gm['std_roc_auc']:.4f}")
    else:
        print("    Mean ROC-AUC:           N/A")

    cd = COLLAPSE_DIAGNOSTICS[model_name]
    print(f"    Single-class folds:     {cd['n_collapsed_single_class']} / {cd['n_folds']} ({cd['collapsed_fraction']:.3f})")
print("=" * 70)